# Redis — clase práctica

El hilo de los ejemplos es **Tienda Sur**. La clienta es Laura Gómez (`cliente:1042`). Los pedidos, el stock y la sesión cuelgan de ese caso.

## 🎯 Objetivos
- Conectarse a Redis desde Python.
- Usar strings, listas, sets y hashes, y completar los ejercicios.
- Juntar comandos en una operación atómica.
- Emitir, mirar y revocar un access token.


## 🔌 Conexión

`decode_responses=True` hace que las respuestas lleguen como `str`. `ping()` responde `True` si el servidor de la clase está en marcha.

`alumno` es el prefijo de las claves propias. Al correr la conexión se genera solo, con la forma `alumno-XY`. Una clave se arma `f"{alumno}:cliente:1042:nombre"`.


In [1]:
%pip install redis

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 560.6/560.6 kB 30.3 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.


In [2]:
import random
import redis

alumno = f"alumno-{random.randint(0, 9999):04d}"
print(alumno)

#r = redis.Redis( host="localhost", port=6379, decode_responses=True)
r = redis.Redis(host="157.92.26.78", port=6379, decode_responses=True)

r.ping()


alumno-7472


True

## Sección 1. Strings, listas, sets y hashes

### Ejercicios resueltos


#### 🔤 Strings. SET y GET

Un string es un valor por clave. `set` crea la clave o reemplaza el valor. `get` solo lee.

En la tienda, el nombre de la clienta vive en su propia clave.


In [3]:
r.set(f"{alumno}:cliente:1042:nombre", "Santiago Drelewicz")
r.get(f"{alumno}:cliente:1042:nombre")  # "Santiago Drelewicz"


'Santiago Drelewicz'

#### 🗑️ Strings. EXISTS y DELETE

`exists` responde `1` si la clave está y `0` si no. `delete` devuelve cuántas claves borró. Los dos comandos valen para cualquier tipo. En redis-py la respuesta es `1` o `0`, no un booleano.


In [4]:
r.set(f"{alumno}:pedido:8841:estado", "pago")
print(r.exists(f"{alumno}:pedido:8841:estado"))
print(r.exists(f"{alumno}:pedido:9999:estado"))
print(r.delete(f"{alumno}:pedido:8841:estado"))


1
0
1


#### 🔢 Strings. INCR

El ejemplo cuenta los pedidos que entraron hoy.

`incrby` suma de a varios y `decr` resta uno. Redis guarda el número como texto: `get` devuelve `"7"`. Para calcular en Python, `int(r.get(...))`.


In [5]:
r.set(f"{alumno}:pedidos:hoy", 5)
r.incrby(f"{alumno}:pedidos:hoy", 3)  # 8
r.decr(f"{alumno}:pedidos:hoy")       # 7
print(r.get(f"{alumno}:pedidos:hoy"))  # "7"


7


#### ⏱️ Strings. TTL

Una reserva de retiro en el local vence sola. `ex` en el `set` le pone los segundos de vida. `ttl` devuelve los segundos que faltan (`N`), `-1` si la clave existe y no vence, y `-2` si no está. `expire` cambia el plazo. `persist` se lo saca.


In [6]:
r.set(f"{alumno}:reserva:turno-19", "cliente:1042", ex=60)
print("N:", r.ttl(f"{alumno}:reserva:turno-19"))
r.expire(f"{alumno}:reserva:turno-19", 30)
print("después de expire:", r.ttl(f"{alumno}:reserva:turno-19"))
r.persist(f"{alumno}:reserva:turno-19")
print("persist:", r.ttl(f"{alumno}:reserva:turno-19"))
print("no está:", r.ttl(f"{alumno}:reserva:inexistente"))


N: 60
después de expire: 30
persist: -1
no está: -2


#### 📋 Listas

La cola de despacho respeta el orden de llegada. `rpush` entra por la derecha. `lpop` saca el pedido que está primero. `lrange` con `0, -1` lee la cola completa. Los ids vuelven como strings.


In [ ]:
r.delete(f"{alumno}:cola:despacho")
r.rpush(f"{alumno}:cola:despacho", "8841", "8842", "8843")
print(r.lrange(f"{alumno}:cola:despacho", 0, -1))
print(r.lpop(f"{alumno}:cola:despacho"))


['8841', '8842', '8843']
8841


#### 🧩 Sets

`clientes:online` guarda quién tiene la sesión abierta. `sadd` no duplica: el `1042` entra una sola vez. `sismember` responde `1` si ese cliente está y `0` si no.


In [11]:
r.delete(f"{alumno}:clientes:online")
r.sadd(f"{alumno}:clientes:online", "1042", "2201", "1042")
print(r.smembers(f"{alumno}:clientes:online"))
print(r.sismember(f"{alumno}:clientes:online", "1042"))


{'2201', '1042'}
1


#### 🗂️ Hashes

La ficha de la clienta es una sola clave y varios fields. Lo que en Python es la key del dict, en Redis es el field. `hget` lee uno. `hgetall` lee la ficha. Cambiar un field deja el resto igual.


In [12]:
r.delete(f"{alumno}:cliente:1042")
r.hset(f"{alumno}:cliente:1042", mapping={
    "nombre": "Laura Gómez",
    "email": "laura.gomez@tiendasur.com",
    "plan": "plus",
})
print(r.hget(f"{alumno}:cliente:1042", "email"))
print(r.hgetall(f"{alumno}:cliente:1042"))


laura.gomez@tiendasur.com
{'nombre': 'Laura Gómez', 'email': 'laura.gomez@tiendasur.com', 'plan': 'plus'}


### Ejercicios para resolver


#### 🔤 Strings

1. Con `set`, guardar `f"{alumno}:cliente:1042:email"` = `laura.gomez@tiendasur.com`. Leerla con `get`.
2. Con un solo `mset`, guardar `f"{alumno}:cliente:1042:ciudad"` = `CABA`, `f"{alumno}:cliente:1042:plan"` = `plus` y `f"{alumno}:cliente:1042:telefono"` = `11-5555-0142`.
3. Leer esas tres juntas con `mget`, en ese orden.
4. Con `append`, agregar ` (interior)` a la ciudad.
5. Imprimir el largo del email con `strlen`.


In [13]:
# Completar
r.set(f"{alumno}:cliente:1042:email", "laura.gomez@tiendasur.com")
print(r.get(f"{alumno}:cliente:1042:email"))

laura.gomez@tiendasur.com


In [23]:
info = {"ciudad": "CABA", "plan": "plus", "telefono": "11-3040-8214"}

r.mset({f"{alumno}:cliente:1042:{key}": info[key] for key in info})
print(r.mget([f"{alumno}:cliente:1042:{key}" for key in info]))

['CABA', 'plus', '11-3040-8214']


In [24]:
r.append(f"{alumno}:cliente:1042:ciudad", "(interior)")
r.get(f"{alumno}:cliente:1042:ciudad")

'CABA(interior)'

In [25]:
r.strlen(f"{alumno}:cliente:1042:email")

25

#### 🗑️ EXISTS y DELETE

Crear `f"{alumno}:envio:552:estado"` = `en_camino` y `f"{alumno}:envio:553:estado"` = `preparando`. Borrarlas en un solo `delete` e imprimir cuántas claves eliminó. Después imprimir `exists` de `f"{alumno}:envio:552:estado"`: tiene que dar `0`.


In [ ]:
# Completar


#### 🔢 INCR

La home de la tienda lleva el contador `f"{alumno}:visitas:home"`.

1. Borrar la clave si ya existe.
2. Sumar una visita con `incr`, otra con `incr` y diez con `incrby`.
3. Restar 4 con `decrby` (visitas de un bot que se descartan).
4. Imprimir el texto que devuelve `get` y el entero. El entero es 8.


In [ ]:
# Completar


#### ⏱️ TTL

La remera negra entra en oferta flash.

1. Guardar `f"{alumno}:oferta:remera-negra"` = `30%` con 60 segundos de vida e imprimir el `ttl`.
2. Extender el plazo a 120 segundos con `expire` y volver a imprimir el `ttl`.
3. Dejar la oferta sin vencimiento con `persist` e imprimir el `ttl` (`-1`).
4. Imprimir el `ttl` de `f"{alumno}:oferta:no-existe"` (`-2`).


In [ ]:
# Completar


#### 📋 Listas

La cocina tiene una cola propia, `f"{alumno}:cola:cocina"`.

1. Con un solo `lpush`, cargar los pedidos `8901`, `8902` y `8903`. Mostrar `lrange`: el último argumento queda al frente.
2. Sacar el último con `rpop` e imprimirlo.
3. Imprimir cuántos quedan (`llen`) y el que quedó al frente (`lindex` 0).


In [ ]:
# Completar


#### 🧩 Sets

1. Sacar al cliente `2201` de `f"{alumno}:clientes:online"` con `srem`. Imprimir `scard` y `sismember` de `2201` (`0`).
2. Crear `f"{alumno}:clientes:vip"` con `1042` y `3300`.
3. Imprimir quién está en los dos conjuntos (`sinter`) y la unión (`sunion`).


In [ ]:
# Completar


#### 🗂️ Hashes

1. Agregar solo el field `ciudad` = `CABA` en `f"{alumno}:cliente:1042"`. Leer la ficha con `hgetall`: nombre, email y plan tienen que seguir.
2. Preguntar con `hexists` si `plan` está. Borrarlo con `hdel`.
3. Imprimir `hkeys` y `hvals`.


In [ ]:
# Completar


## Sección 2. Operaciones atómicas

### Un comando

`set` con `nx=True` y `ex=30` hace las dos cosas en el mismo comando: crea `checkout:1042` solo si nadie está pagando, y le pone 30 segundos de vida. Redis termina ese comando antes de atender a otro cliente.

### Varios comandos

Guardar el carrito y ponerle vencimiento son dos comandos. Si se mandan por separado, entre el `hset` y el `expire` puede colarse otra operación y el carrito quedar guardado sin plazo.

`pipeline()` junta esos comandos en el cliente y los manda de una vez con `execute()`.

- **`pipeline()`**, sin argumentos, abre una transacción. Redis los envuelve en `MULTI` / `EXEC`: aplica el `hset` y el `expire` seguidos, y devuelve una lista con el resultado de cada uno.
- **`pipeline(transaction=False)`** también los manda en un solo envío, y por eso ahorra viajes de red. Entre el primer comando y el segundo, Redis puede atender a otro cliente. Sirve para cargar muchos datos. Para el carrito con vencimiento hace falta la transacción.


### Ejemplo


In [26]:
r.delete(f"{alumno}:checkout:1042")
print(r.set(f"{alumno}:checkout:1042", "1", nx=True, ex=30))

r.delete(f"{alumno}:carrito:1042")
pipe = r.pipeline()
pipe.hset(f"{alumno}:carrito:1042", mapping={"sku": "remera-negra", "cantidad": "1"})
pipe.expire(f"{alumno}:carrito:1042", 900)
print(pipe.execute())
print(r.hgetall(f"{alumno}:carrito:1042"))


True
[2, True]
{'sku': 'remera-negra', 'cantidad': '1'}


### Ejercicio

Laura aparta dos remeras.

1. Con un `pipeline()` (transacción), guardar en `f"{alumno}:reserva:1042"` los fields `sku` = `remera-negra` y `cantidad` = `2`, y ponerle 300 segundos de vida.
2. Después de `execute`, imprimir `hgetall` y `ttl`.
3. Repetir el `set` de `f"{alumno}:checkout:1042"` con `nx=True` y `ex=30`. Imprimir el resultado: la clave ya existe, así que vuelve `None`.


In [ ]:
# Completar


## Sección 3. Access token

La tienda web le entrega a Laura un token opaco. En Redis la clave es `f"{alumno}:token:{token}"`. El hash guarda `sub`, `client` y `scope`, con 900 segundos de vida. Si la clave no está, la sesión no está activa. `delete` cierra la sesión sin esperar el vencimiento.

### Ejemplo


In [27]:
import secrets

token = secrets.token_urlsafe(32)
key = f"{alumno}:token:{token}"
datos = {"sub": "cliente:1042", "client": "tienda-web", "scope": "orders:read"}

pipe = r.pipeline()
pipe.hset(key, mapping=datos)
pipe.expire(key, 900)
pipe.execute()

data = r.hgetall(key)
resp = {"active": False} if not data else {"active": True, **data}
print(key)
print(resp)

r.delete(key)
print(r.hgetall(key))  # {}


alumno-7472:token:EbLvwNgjPJO9rdB3LF6Piuilm3wozKofEWJ0cnpUnnw
{'active': True, 'sub': 'cliente:1042', 'client': 'tienda-web', 'scope': 'orders:read'}
{}


### Ejercicio

Laura pasa a cargar un pedido y la tienda le cambia el alcance.

1. Emitir un token nuevo con `scope` = `orders:write` y TTL de 60 segundos. La clave es `f"{alumno}:token:{token}"`.
2. Armar la introspección e imprimirla: `active` en `True` y el scope `orders:write`. Imprimir también el `ttl`.
3. Revocar la clave con `delete`. Volver a armar la introspección: `active` queda en `False`. Imprimir `exists` de la clave (`0`).


In [ ]:
# Completar


## 📎 Anexo — Hash frente a lista

Queda fuera de la clase. Compara, con 5.000 elementos y un pipeline sin transacción, la escritura, la lectura completa y la búsqueda (`hget` frente a `lpos`).


In [28]:
import time

N = 5_000
r.delete(f"{alumno}:mi_hash", f"{alumno}:mi_lista")

def medir(pipe):
    inicio = time.perf_counter()
    pipe.execute()
    return time.perf_counter() - inicio

pipe = r.pipeline(transaction=False)
for i in range(N):
    pipe.hset(f"{alumno}:mi_hash", f"campo_{i}", f"valor_{i}")
hash_write = medir(pipe)

pipe = r.pipeline(transaction=False)
for i in range(N):
    pipe.rpush(f"{alumno}:mi_lista", f"valor_{i}")
list_write = medir(pipe)

inicio = time.perf_counter()
r.hgetall(f"{alumno}:mi_hash")
hash_read_all = time.perf_counter() - inicio

inicio = time.perf_counter()
r.lrange(f"{alumno}:mi_lista", 0, -1)
list_read_all = time.perf_counter() - inicio

pipe = r.pipeline(transaction=False)
for i in range(N):
    pipe.hget(f"{alumno}:mi_hash", f"campo_{i}")
hash_find = medir(pipe)

pipe = r.pipeline(transaction=False)
for i in range(N):
    pipe.lpos(f"{alumno}:mi_lista", f"valor_{i}")
list_find = medir(pipe)

r.delete(f"{alumno}:mi_hash", f"{alumno}:mi_lista")

print(f"{'':22} {'Hash':>10} {'Lista':>10}")
print(f"{'Escritura':22} {hash_write:10.4f} {list_write:10.4f}")
print(f"{'Lectura completa':22} {hash_read_all:10.4f} {list_read_all:10.4f}")
print(f"{'Búsqueda':22} {hash_find:10.4f} {list_find:10.4f}")


                             Hash      Lista
Escritura                  0.0495     0.0380
Lectura completa           0.0211     0.0108
Búsqueda                   0.0366     0.5248


## 🧹 Limpieza

El bloque de abajo borra las claves de este `alumno`. Usarlo solo en el Redis de la clase.


In [ ]:
# Borra solo las claves de este alumno.
# for key in r.keys(f"{alumno}:*"):
#     r.delete(key)
